# Velocity Limit Validation (Colab Ready)

This notebook tests temporal velocity limits (v_max = 2.0 units/frame) with real images or videos.

## Setup
Run the following cells to install dependencies and set up the environment.

In [ ]:
# Install dependencies
!pip install torch controlnet-aux opencv-python-headless pillow numpy matplotlib
!pip install insightface onnxruntime

In [ ]:
# Mount Google Drive (optional - for loading custom videos)
from google.colab import drive
drive.mount('/content/gdrive')

## Load Libraries

In [ ]:
import sys
sys.path.append('/content/pivot' if __import__('os').path.exists('/content/pivot') else '..')

import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

from core.kinematic_guardrail import (
    PoseEstimator,
    compute_velocity_loss,
    compute_l_physics,
    V_MAX_DEFAULT,
    COCO_KEYPOINTS,
    COCO_BONES
)

print(f"v_max default: {V_MAX_DEFAULT} units/frame")
print("Libraries loaded successfully")

## Option A: Test with Video File

In [ ]:
# Test with video file
VIDEO_PATH = '/content/gdrive/MyDrive/videos/test_video.mp4'  # Change to your video path

def extract_frames_from_video(video_path, max_frames=30):
    """Extract frames from video file."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    cap.release()
    return frames

def estimate_poses_from_frames(frames, pose_estimator):
    """Estimate pose keypoints from list of frames."""
    keypoints_list = []
    
    for frame in frames:
        keypoints = pose_estimator.estimate_pose(frame)
        keypoints_list.append(keypoints)
    
    return np.array(keypoints_list)

# Run video test
if Path(VIDEO_PATH).exists():
    print(f"Loading video: {VIDEO_PATH}")
    
    # Initialize pose estimator
    pose_estimator = PoseEstimator(device='cuda')
    
    # Extract frames
    frames = extract_frames_from_video(VIDEO_PATH, max_frames=30)
    print(f"Extracted {len(frames)} frames")
    
    # Estimate poses
    print("Estimating poses...")
    keypoints = estimate_poses_from_frames(frames, pose_estimator)
    print(f"Pose keypoints shape: {keypoints.shape}")
    
    # Compute velocity loss
    pose_array = keypoints[:, :, :2][None]  # Add batch dim [B, T, K, 2]
    velocities, loss = compute_velocity_loss(pose_array, v_max=2.0)
    
    print(f"\nVelocity Analysis:")
    print(f"  Max velocity: {np.max(velocities):.2f} units/frame")
    print(f"  Mean velocity: {np.mean(velocities):.2f} units/frame")
    print(f"  Velocity loss: {loss:.4f}")
    print(f"  v_max: {V_MAX_DEFAULT}")
    
    # Check if valid
    is_valid = np.max(velocities) <= V_MAX_DEFAULT
    print(f"\n{'✓ VALID' if is_valid else '✗ INVALID'}: Velocity within limits")
else:
    print(f"Video not found: {VIDEO_PATH}")
    print("Skipping video test. Use Option B instead.")

## Option B: Test with Image Sequence

In [ ]:
# Test with image sequence
IMAGE_DIR = '/content/gdrive/MyDrive/images/'  # Change to your image directory

def load_image_sequence(image_dir, max_frames=30):
    """Load images from directory."""
    image_dir = Path(image_dir)
    if not image_dir.exists():
        return []
    
    image_files = sorted(image_dir.glob('*.jpg')) + sorted(image_dir.glob('*.png'))
    frames = []
    
    for f in image_files[:max_frames]:
        img = cv2.imread(str(f))
        if img is not None:
            frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    
    return frames

# Run image sequence test
if Path(IMAGE_DIR).exists():
    print(f"Loading images from: {IMAGE_DIR}")
    
    # Initialize pose estimator
    pose_estimator = PoseEstimator(device='cuda')
    
    # Load images
    frames = load_image_sequence(IMAGE_DIR, max_frames=30)
    print(f"Loaded {len(frames)} frames")
    
    if len(frames) > 0:
        # Estimate poses
        print("Estimating poses...")
        keypoints = estimate_poses_from_frames(frames, pose_estimator)
        print(f"Pose keypoints shape: {keypoints.shape}")
        
        # Compute velocity loss
        pose_array = keypoints[:, :, :2][None]
        velocities, loss = compute_velocity_loss(pose_array, v_max=2.0)
        
        print(f"\nVelocity Analysis:")
        print(f"  Max velocity: {np.max(velocities):.2f} units/frame")
        print(f"  Mean velocity: {np.mean(velocities):.2f} units/frame")
        print(f"  Velocity loss: {loss:.4f}")
        
        is_valid = np.max(velocities) <= V_MAX_DEFAULT
        print(f"\n{'✓ VALID' if is_valid else '✗ INVALID'}: Velocity within limits")
    else:
        print("No images found")
else:
    print(f"Image directory not found: {IMAGE_DIR}")
    print("Skipping image test. Use Option C instead.")

## Option C: Test with Generated Motion

In [ ]:
# Test with generated synthetic motion
print("Testing with synthetic motion...")

# Create base pose
base = np.stack([
    np.linspace(256, 256, 17),  # x coordinates
    np.linspace(100, 400, 17)  # y coordinates
], axis=-1).astype(np.float32)

# Test 1: Slow motion (valid)
slow_motion = np.stack([base, base + [1.0, 0.0]], axis=0)[None]
velocities, loss = compute_velocity_loss(slow_motion, v_max=2.0)
print(f"Slow motion (1.0): max_v={np.max(velocities):.2f}, loss={loss:.4f} {'✓' if loss == 0 else '✗'}")

# Test 2: Medium motion (valid)
medium_motion = np.stack([base, base + [1.5, 0.0]], axis=0)[None]
velocities, loss = compute_velocity_loss(medium_motion, v_max=2.0)
print(f"Medium motion (1.5): max_v={np.max(velocities):.2f}, loss={loss:.4f} {'✓' if loss == 0 else '✗'}")

# Test 3: Fast motion (invalid)
fast_motion = np.stack([base, base + [3.0, 0.0]], axis=0)[None]
velocities, loss = compute_velocity_loss(fast_motion, v_max=2.0)
print(f"Fast motion (3.0): max_v={np.max(velocities):.2f}, loss={loss:.4f} {'✓' if loss == 0 else '✗'}")

print("\n✓ Synthetic motion tests complete")

## Full Verification Test

In [ ]:
# Full verification with L_physics composite
print("Testing L_physics composite loss...")

from core.verification_daemon import create_verification_daemon

# Create verification daemon
daemon = create_verification_daemon(
    v_max=2.0,
    kinematic_threshold=1.0,
    enable_kinematic=True,
    enable_logging=True
)

# Test with slow motion
result = daemon.verify_kinematic(slow_motion)
print(f"\nSlow motion: passed={result.passed}, loss={result.total_loss:.4f}")

# Test with fast motion
result = daemon.verify_kinematic(fast_motion)
print(f"Fast motion: passed={result.passed}, loss={result.total_loss:.4f}")

print("\n✓ Full verification test complete")

## Visualization

In [ ]:
# Visualize velocity over time
print("Creating velocity visualization...")

# Create multi-frame pose sequence
poses = []
for t in range(10):
    offset = t * 0.8  # Slow motion
    pose = base + [offset, 0]
    poses.append(pose)
pose_sequence = np.stack(poses, axis=0)[None]

# Compute velocities
velocities, _ = compute_velocity_loss(pose_sequence, v_max=2.0)
velocities = velocities[0]  # Remove batch dim

# Plot
plt.figure(figsize=(12, 4))
plt.plot(np.max(velocities, axis=1), label='Max velocity')
plt.axhline(y=2.0, color='r', linestyle='--', label='v_max=2.0')
plt.xlabel('Frame')
plt.ylabel('Velocity (units/frame)')
plt.title('Velocity Over Time - Slow Motion')
plt.legend()
plt.grid(True)
plt.show()

# Create fast motion sequence
poses_fast = []
for t in range(10):
    offset = t * 2.5  # Fast motion
    pose = base + [offset, 0]
    poses_fast.append(pose)
pose_sequence_fast = np.stack(poses_fast, axis=0)[None]

velocities_fast, _ = compute_velocity_loss(pose_sequence_fast, v_max=2.0)
velocities_fast = velocities_fast[0]

plt.figure(figsize=(12, 4))
plt.plot(np.max(velocities_fast, axis=1), label='Max velocity')
plt.axhline(y=2.0, color='r', linestyle='--', label='v_max=2.0')
plt.xlabel('Frame')
plt.ylabel('Velocity (units/frame)')
plt.title('Velocity Over Time - Fast Motion')
plt.legend()
plt.grid(True)
plt.show()

print("✓ Visualization complete")

## Summary

This notebook validates velocity limits with:
- Option A: Video file input
- Option B: Image sequence input
- Option C: Synthetic motion
- Full verification daemon integration

Results:
- v_max = 2.0 units/frame
- Velocity loss penalizes motion exceeding v_max
- L_physics composite combines bone + velocity constraints